# N3 Slinky Sim Max-Dlambda Ablation

Runs max-dlambda ablations on the 3-node simulated slinky data using the same core settings as `2d_slinky_train_from_sim_LLT.ipynb`. Training remains strict on nonconvergence, while validation and final prediction remain non-strict. All outputs, per-case plots, summaries, and optional post-training Hessian diagnostics are collected under one `OUTPUT_DIR`.

In [ ]:
import csv
import json
import os
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np

jax.config.update("jax_enable_x64", True)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from properties import SlinkyN3Properties
from run_max_dlambda_ablation_minimal import MaxDlambdaAblationConfig

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "max_dlambda_ablation_outputs_n3_slinky_sim"
SUMMARY_DIR = OUTPUT_DIR / "max_dlambda_ablation_summary"

train_file = "../simulation_data_2D/3_noded/n3_slinky_sim_train_dataset_7_trajs.npz"
valid_file = "../simulation_data_2D/3_noded/n3_slinky_sim_test_dataset_6_trajs.npz"

properties = SlinkyN3Properties(mass=0.2)
K_init_chol = (0.02, 0.0, 0.05)
K_init_diag = (0.02, 0.05)

cfg = MaxDlambdaAblationConfig(
    der_K_diag=K_init_diag,
    der_K_chol=K_init_chol,
    hidden=(10, 10),
    corr_factor=0.05,
    input_mode="invariant",
    only_stretching_NN=True,
    only_bending_NN=False,
    zero_reference=True,
    activation="tanh",
    n_epochs=500,
    lr=1e-3,
    seed_list=(42,),
    valid_every=1,
    max_dlambda_values=(1e-3, 5e-3, 1e-2, 5e-2, 1e-1, 5e-1, 1.0),
    iters=20,
    ls_steps=10,
    abs_tol=1e-4,
    rel_tol=1e-4,
    early_stop=True,
    train_fail_on_nonconvergence=True,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=1e-4,
    hessian_reg_probes=1,
    hessian_reg_seed=0,
    force_key="F",
    force_loss_strength=0.1,
    force_components=(0,),
    force_sign=1.0,
    return_loss_components=True,
    early_stopping=True,
    early_stopping_patience=200,
    early_stopping_min_delta=1e-5,
    restore_best_model=True,
    output_dir=str(OUTPUT_DIR),
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_force_predictions=True,
    plot_force_predictions=True,
    save_hessian_diagnostics=False,
    save_summary_json=True,
    strict_finite_check=True,
    stop_after_first_failure=False,
    verbose=True,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving max-dlambda ablation results under: {OUTPUT_DIR.resolve()}")

Saving max-dlambda ablation results under: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/max_dlambda_ablation_outputs_n3_slinky_sim


In [ ]:
from run_max_dlambda_ablation_minimal import subset_all, subset_main_paper_candidates

# Default: focused ablation on the main comparison candidates.
selected_architectures = subset_main_paper_candidates()
# selected_architectures = [
#     "diag_energy_baseline",
#     "chol_energy_baseline",
# ]

# To run every registered architecture, use:
# selected_architectures = subset_all()

print(f"Running {len(selected_architectures)} architectures across max_dlambda values {cfg.max_dlambda_values}:")
for name in selected_architectures:
    print(f"  - {name}")

print("\nTraining fail_on_nonconvergence:", cfg.train_fail_on_nonconvergence)
print("Prediction fail_on_nonconvergence:", cfg.prediction_fail_on_nonconvergence)

Running 2 architectures across max_dlambda values (0.1, 1.0):
  - diag_energy_baseline
  - chol_energy_baseline

Training fail_on_nonconvergence: True
Prediction fail_on_nonconvergence: False


In [ ]:
from run_max_dlambda_ablation_minimal import run_max_dlambda_ablation

all_results = run_max_dlambda_ablation(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    cfg=cfg,
    selected_architectures=selected_architectures,
)

Architecture : diag_energy_baseline
which_case   : baseline
seed         : 42
max_dlambda  : 1.000e-01
iters        : 20
ls_steps     : 10
abs_tol      : 1.000e-04
rel_tol      : 1.000e-04
early_stop   : True
training fail_on_nonconvergence      : True
validation loss fail_on_nonconvergence: False
prediction fail_on_nonconvergence    : False
early_stopping                       : True
early_stopping_patience              : 200
restore_best_model                   : True
hessian_reg_strength                 : 1.000e-04
hessian_reg_probes                   : 1
hessian_reg_seed                     : 0
force_key                            : F
force_loss_strength                  : 1.000e-01
force_components                     : (0,)
force_sign                           : 1
Epoch 000 | Train total: 1.313e-01 | Train disp: 3.253e-02 | Train force: 9.520e-01 | Valid total: 1.052e-01 | Valid disp: 2.705e-02 | Valid force: 7.819e-01
Epoch 100 | Train total: 1.187e-01 | Train disp: 2.704e-02 | 

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/architecture_plots.py:234: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


[diag_energy_baseline] seed=42 | max_dlambda=1.000e-01 | train=9.792e-02 | valid=7.569e-02 | SUCCESS
Architecture : diag_energy_baseline
which_case   : baseline
seed         : 42
max_dlambda  : 1.000e+00
iters        : 20
ls_steps     : 10
abs_tol      : 1.000e-04
rel_tol      : 1.000e-04
early_stop   : True
training fail_on_nonconvergence      : True
validation loss fail_on_nonconvergence: False
prediction fail_on_nonconvergence    : False
early_stopping                       : True
early_stopping_patience              : 200
restore_best_model                   : True
hessian_reg_strength                 : 1.000e-04
hessian_reg_probes                   : 1
hessian_reg_seed                     : 0
force_key                            : F
force_loss_strength                  : 1.000e-01
force_components                     : (0,)
force_sign                           : 1
Epoch 000 | Train total: 1.313e-01 | Train disp: 3.253e-02 | Train force: 9.520e-01 | Valid total: 1.052e-01 | Valid d

In [ ]:
from run_max_dlambda_ablation_minimal import summarize_best_stable_step

SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

flat_records = [record for records in all_results.values() for record in records]
csv_path = SUMMARY_DIR / "max_dlambda_records.csv"
json_path = SUMMARY_DIR / "best_stable_step_summary.json"

if flat_records:
    fieldnames = sorted({key for record in flat_records for key in record.keys()})
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(flat_records)

best_stable_summary = summarize_best_stable_step(all_results)
with open(json_path, "w") as f:
    json.dump(best_stable_summary, f, indent=2)

# Summary plot: final validation loss versus max_dlambda.
fig, ax = plt.subplots(figsize=(8.5, 5.5))
for arch_name, records in all_results.items():
    records_sorted = sorted(records, key=lambda r: (int(r["seed"]), float(r["max_dlambda"])))
    seeds = sorted(set(int(r["seed"]) for r in records_sorted))
    for seed in seeds:
        seed_records = [r for r in records_sorted if int(r["seed"]) == seed]
        xs = np.asarray([r["max_dlambda"] for r in seed_records], dtype=float)
        ys = np.asarray([r["final_valid_loss"] if r["success"] else np.nan for r in seed_records], dtype=float)
        label = arch_name if seed == seeds[0] else None
        ax.plot(xs, ys, marker="o", linewidth=1.8, label=label)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("max_dlambda")
ax.set_ylabel("Final validation loss")
ax.set_title("Max-dlambda ablation")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
summary_plot = SUMMARY_DIR / "final_valid_loss_vs_max_dlambda.png"
fig.savefig(summary_plot, dpi=300, bbox_inches="tight")
plt.close(fig)

print("Wrote:")
print(" ", csv_path.resolve())
print(" ", json_path.resolve())
print(" ", summary_plot.resolve())

In [ ]:
from run_max_dlambda_ablation_minimal import run_max_dlambda_hessian_diagnostics

# Post-training Hessian diagnostics. This is separate from training so the
# max-dlambda sweep stays focused on convergence/training behavior.
HESSIAN_USE_PREDICTED = True
HESSIAN_STRIDE = 10
HESSIAN_MAX_TRAJECTORIES = 1

run_max_dlambda_hessian_diagnostics(
    str(OUTPUT_DIR),
    use_predicted=HESSIAN_USE_PREDICTED,
    splits=("train", "valid"),
    stride=HESSIAN_STRIDE,
    max_trajectories=HESSIAN_MAX_TRAJECTORIES,
    all_trajectories=False,
    fail_on_nonconvergence=False,
)

In [ ]:
print("All max-dlambda ablation artifacts are organized under:")
print(" ", OUTPUT_DIR.resolve())
print("\nMain subfolders/files to inspect:")
print("  - <architecture>/seed_<seed>/<run_name>/results.npz")
print("  - <architecture>/seed_<seed>/<run_name>/model.eqx")
print("  - <architecture>/seed_<seed>/<run_name>/loss_curves.png")
print("  - <architecture>/seed_<seed>/<run_name>/force_pred_vs_truth_*.png")
print("  - <architecture>/seed_<seed>/<run_name>/hessian_diagnostics_*.npz")
print("  - max_dlambda_ablation_summary/")